# Emerging Technologies

In [1]:
import numpy as np
import random
import itertools as it
from IPython.display import Latex

## Problem 1: Generating Random Boolean Functions


In the [Deutsch-Jozsa problem](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/quantum-query-algorithms/deutsch-jozsa-algorithm?utm_source=chatgpt.com), we are given [black-box (oracle)](https://quantumcomputing.stackexchange.com/questions/4625/what-exactly-is-an-oracle/4626#4626) access to a function (meaning our algorithm doesnt get to see how `f` was built, we can only query inputs and observe outputs):

<img src="images/boolean_function.png" width="150">

with a promise that `f` is either:
* constant: same output for every input, or
* balanced: outputs 0 for exactly half the inputs and 1 for the other half.

The task is to decide which one `f` is, using as few calls (queries) to f as possible.

From this, to simulate this problem classicaly we write `random_constant_balanced(n)` which returns an oracle function that satisfies this promise:

This is done by firstly building a truth table `f_a` of length $2^n$ (one output for every n-bit input). It then returns a closure f(*args) that:
1. takes an n-bit input eg. (0, 0, 0, 1),
2. converts it into an index from 0 to $2^n - 1$,
3. returns the output stored in the truth table at that index.

`random.choice` is only used to randomly select which promised case (constant or balanced) we generate for testing. The oracle itself is the returned closure `f`, since our algorithm can only query it via calls like `f(x)`.

This all allows us to test a classical decision procedure (and later compare to the [Deutsch-Jozsa quantam algorithm](https://quantum.cloud.ibm.com/learning/en/modules/computer-science/deutsch-jozsa)) using a function that matches the promised structure IBM discusses.


In [2]:
def random_constant_balanced(n):
    """
    Return a random constant or balanced Boolean function with n boolean inputs.
        
       Returned function will be:
       - constant: outputs all 1's or all 0's for the 2**n inputs, or
       - balanced: outputs True for 50% of the 2**n inputs and False for the other 50%
    
    This is done by generating a truth-table output column (length 2**n) and returning a closure
    that maps input bits to the correct truth-table entry   
    """ 
    #Building truth table table outputs for unknown function
    #Function will be choice between constant or balanced, 50/50
    if random.choice([True, False]):  #constant case
        #Picking the constant output value (0 or 1) and repeat it 2**n times eg (e.g. (1,) * 4 --> (1, 1, 1, 1))
        f_a = (random.choice([0, 1]),) * (2**n)
    else:  #balanced
        #List with half 0's, half 1's (in that order).
        t = [0] * (2**(n-1)) + [1] * (2**(n-1))
        #Shuffle bits so that the 1's and 0's are randomly placed in the tuple
        random.shuffle(t)
        f_a = tuple(t) #Make it a tuple

    #Return function that uses truth table (closure)
    def f(*args):
        """Function takes n binary arguments and converts them to an int.
          Returning 0/1."""
        
        #If caller gives length bigger than n, return error
        if len(args) != n:
            raise ValueError(f"Wrong number of arguments")
        #Accepts any truthy values for a '1' and likewise for a '0' and converts to a binary string eg "1001"
        #String then converted to integer index (base 2 - 0 or 1)
        bits_int = int(''.join('1' if i else '0' for i in args), 2)
        return f_a[bits_int]#Look up and return the output bit from the truth table

    return f #Return closure so caller gets callable function f(...)

In [3]:
#Testing f
f = random_constant_balanced(4)
f(0, 0, 0, 1)

1

## Problem 2: Classical Testing for Function Type


In [Deutsch-Jozsa](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/quantum-query-algorithms/deutsch-jozsa-algorithm?utm_source=chatgpt.com), we only get oracle (black-box) access to a Boolean function `f`. That means we cannot inspect how `f` is built, we can only query inputs and observe outputs. Before we see why the quantum algorithm is powerful, we first need to understand how many oracle queries a classical algorithm needs to decide whether `f` is constant or balanced.

We are given:

<img src="images/boolean_function.png" width="150">

with a promise that `f` is either:
* constant: same output for every input, or
* balanced: outputs 0 for exactly half the inputs and 1 for the other half.

**The Classical algorithm used:**

To determine if `f` is constant or balanced:
1. Evalute `f` on one input and store the output.
2. Evalute `f` on the rest of the inputs and compare their outputs to the first.
3. If any output differs, return "balanced" immediately.
4. If all outputs match, return "constant".

This is correct under the promise: seeing both 0 and 1 cannot happen for a constant function, so it must be balanced.


**Efficiency of solution**

We evaluate `f` on binary input tuples of length `n`, store the first output, and compare the subsequent outputs to it. If we ever see a different output, we can immediately return "balanced", so the algorithm can terminate early, making this is a deterministic classical algorithm.

In the worst case, to be 100% certain we must call `f`, $2^{n-1} + 1$ times: a balanced function outputs 0 on exactly half the inputs and 1 on the other half (ie. $2^{n-1}$ zeros and $2^{n-1}$ ones). Therefore it’s possible to see the same output for the first $2^{n-1}$ queries and still not know if `f` is constant or balanced. The next query resolves it. For `n` = 4, the maximum is $2^3 + 1 = 9$ calls.  

[IBM Quantum Learning notes that a deterministic classical algorithm requires $2^{n-1} + 1$ queries in worst case.](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/quantum-query-algorithms/deutsch-jozsa-algorithm?utm_source=chatgpt.com)

In [ ]:
def determine_constant_balanced(f, n):
    """
    Returns 'constant' if f returns the same value for all 2**n inputs, otherwise return
    'balanced'.

    Done by evaluating f on binary input tuples of length n. It stores the output
    for the first input and then compares all subsequent outputs to it. If any output
    differs, f is 'balanced'; if all outputs match, f is 'constant'.
    """
    
    #Generator for all binary tuples of length n - 2**n in total
    bin_tuples_n = it.product((0, 1), repeat=n)
    #Getting first input tuple
    first_tuple = next(bin_tuples_n) 
    #Calling f on first tuple and storing its output
    first = f(*first_tuple)  
    print(first_tuple, first)

    #Looping through rest of tuples 
    for x in bin_tuples_n:            
        #Getting inputs in current tuple
        val = f(*x)
        print(x, val)
        #If value isnt the same as first tuple, return balanced
        if val != first:
            return "balanced"
    #Otherwise f is constant
    return "constant"

In [5]:
determine_constant_balanced(f, 4)

(0, 0, 0, 0) 1
(0, 0, 0, 1) 1
(0, 0, 1, 0) 1
(0, 0, 1, 1) 1
(0, 1, 0, 0) 1
(0, 1, 0, 1) 1
(0, 1, 1, 0) 1
(0, 1, 1, 1) 1
(1, 0, 0, 0) 1
(1, 0, 0, 1) 1
(1, 0, 1, 0) 1
(1, 0, 1, 1) 1
(1, 1, 0, 0) 1
(1, 1, 0, 1) 1
(1, 1, 1, 0) 1
(1, 1, 1, 1) 1


'constant'

## Problem 3: Quantum Oracles


## Problem 4: Deutsch's Algorithm with Qiskit


## Problem 5: Scaling to the Deutsch–Jozsa Algorithm
